# 模型量化教程 (Model Quantization Tutorial)

本教程详细介绍深度学习模型量化技术，包括：

1. **量化基础**: 理解量化的数学原理
2. **动态量化**: 推理时动态计算量化参数
3. **静态量化**: 使用校准数据预计算量化参数
4. **量化感知训练 (QAT)**: 在训练中模拟量化效果

---

## 为什么需要量化？

| 数据类型 | 位数 | 内存占用 | 计算速度 |
|:---------|:----:|:--------:|:--------:|
| FP32 | 32 | 4 字节 | 基准 |
| FP16 | 16 | 2 字节 | 2x 加速 |
| INT8 | 8 | 1 字节 | 2-4x 加速 |
| INT4 | 4 | 0.5 字节 | 4-8x 加速 |

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch 版本: {torch.__version__}")

## 1. 量化基础

### 1.1 量化公式

量化将浮点数映射到整数：

$$q = \text{round}\left(\frac{r}{s}\right) + z$$

反量化：

$$r = s \cdot (q - z)$$

其中：
- $r$: 原始浮点值
- $q$: 量化后的整数值
- $s$: 缩放因子 (scale)
- $z$: 零点 (zero point)

In [ ]:
from quantization import compute_scale_zero_point, quantize_tensor, dequantize_tensor

# 创建示例数据
x = torch.randn(1000) * 2  # 范围约 [-6, 6]

# 计算量化参数
x_min, x_max = x.min(), x.max()
scale, zero_point = compute_scale_zero_point(
    x_min, x_max, 
    qmin=-128, qmax=127, 
    symmetric=True
)

print(f"原始数据范围: [{x_min:.4f}, {x_max:.4f}]")
print(f"Scale: {scale:.6f}")
print(f"Zero Point: {zero_point}")

In [ ]:
# 量化和反量化
x_quantized = quantize_tensor(x, scale, zero_point, -128, 127)
x_dequantized = dequantize_tensor(x_quantized, scale, zero_point)

# 计算量化误差
error = (x - x_dequantized).abs()

print(f"量化后数据类型: {x_quantized.dtype}")
print(f"量化值范围: [{x_quantized.min()}, {x_quantized.max()}]")
print(f"\n量化误差统计:")
print(f"  平均误差: {error.mean():.6f}")
print(f"  最大误差: {error.max():.6f}")
print(f"  相对误差: {(error / (x.abs() + 1e-8)).mean() * 100:.2f}%")

In [ ]:
# 可视化量化效果
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 原始数据分布
axes[0].hist(x.numpy(), bins=50, alpha=0.7, color='blue')
axes[0].set_title('原始数据分布 (FP32)')
axes[0].set_xlabel('值')
axes[0].set_ylabel('频数')

# 量化后数据分布
axes[1].hist(x_quantized.numpy(), bins=50, alpha=0.7, color='green')
axes[1].set_title('量化后数据分布 (INT8)')
axes[1].set_xlabel('量化值')
axes[1].set_ylabel('频数')

# 量化误差分布
axes[2].hist(error.numpy(), bins=50, alpha=0.7, color='red')
axes[2].set_title('量化误差分布')
axes[2].set_xlabel('误差')
axes[2].set_ylabel('频数')

plt.tight_layout()
plt.show()

### 1.2 对称量化 vs 非对称量化

**对称量化**:
- zero_point = 0
- 范围: [-127, 127]
- 适合权重（通常以 0 为中心）

**非对称量化**:
- zero_point ≠ 0
- 范围: [0, 255] 或 [-128, 127]
- 适合激活值（如 ReLU 输出）

In [ ]:
# 比较对称和非对称量化
# 模拟 ReLU 输出 (非负)
x_relu = F.relu(torch.randn(1000) * 2)

# 对称量化
scale_sym, zp_sym = compute_scale_zero_point(
    x_relu.min(), x_relu.max(), -128, 127, symmetric=True
)
x_q_sym = quantize_tensor(x_relu, scale_sym, zp_sym, -128, 127)
x_dq_sym = dequantize_tensor(x_q_sym, scale_sym, zp_sym)

# 非对称量化
scale_asym, zp_asym = compute_scale_zero_point(
    x_relu.min(), x_relu.max(), 0, 255, symmetric=False
)
x_q_asym = quantize_tensor(x_relu, scale_asym, zp_asym, 0, 255)
x_dq_asym = dequantize_tensor(x_q_asym, scale_asym, zp_asym)

print("ReLU 输出量化比较:")
print(f"\n对称量化:")
print(f"  Scale: {scale_sym:.6f}, Zero Point: {zp_sym}")
print(f"  平均误差: {(x_relu - x_dq_sym).abs().mean():.6f}")

print(f"\n非对称量化:")
print(f"  Scale: {scale_asym:.6f}, Zero Point: {zp_asym}")
print(f"  平均误差: {(x_relu - x_dq_asym).abs().mean():.6f}")

## 2. 动态量化

动态量化在推理时动态计算激活值的量化参数，权重预先量化。

**优点**:
- 无需校准数据
- 实现简单

**缺点**:
- 每次推理都需要计算量化参数
- 精度可能略低于静态量化

In [ ]:
from quantization import DynamicQuantizer, QuantizationConfig, QuantizationType

# 定义测试模型
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, num_classes)
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

# 创建模型
model = SimpleClassifier()
model.eval()

# 计算原始模型大小
def get_model_size(model):
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    return param_size / (1024 * 1024)  # MB

print(f"原始模型大小: {get_model_size(model):.2f} MB")

In [ ]:
# 应用动态量化
quantizer = DynamicQuantizer()
quantized_model = quantizer.quantize(model)

# 测试推理
x_test = torch.randn(32, 784)

with torch.no_grad():
    original_output = model(x_test)
    quantized_output = quantized_model(x_test)

# 比较输出
output_diff = (original_output - quantized_output).abs()
print(f"输出差异:")
print(f"  平均差异: {output_diff.mean():.6f}")
print(f"  最大差异: {output_diff.max():.6f}")

# 检查预测是否一致
original_pred = original_output.argmax(dim=1)
quantized_pred = quantized_output.argmax(dim=1)
accuracy = (original_pred == quantized_pred).float().mean()
print(f"\n预测一致率: {accuracy * 100:.1f}%")

In [ ]:
# 性能比较
import time

def benchmark_model(model, input_tensor, num_runs=100):
    """测量模型推理时间"""
    model.eval()
    
    # 预热
    with torch.no_grad():
        for _ in range(10):
            _ = model(input_tensor)
    
    # 计时
    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.perf_counter()
            _ = model(input_tensor)
            end = time.perf_counter()
            times.append((end - start) * 1000)
    
    return np.mean(times), np.std(times)

x_bench = torch.randn(64, 784)

original_time, original_std = benchmark_model(model, x_bench)
quantized_time, quantized_std = benchmark_model(quantized_model, x_bench)

print(f"推理时间比较 (batch_size=64):")
print(f"  原始模型: {original_time:.3f} ± {original_std:.3f} ms")
print(f"  量化模型: {quantized_time:.3f} ± {quantized_std:.3f} ms")
print(f"  加速比: {original_time / quantized_time:.2f}x")

## 3. 静态量化

静态量化使用校准数据预先计算激活值的量化参数。

**流程**:
1. 准备校准数据集
2. 运行模型收集激活值统计
3. 计算量化参数
4. 量化模型

In [ ]:
from quantization import StaticQuantizer, calibrate_model

# 创建新模型
model_static = SimpleClassifier()
model_static.eval()

# 创建校准数据
calibration_data = [torch.randn(32, 784) for _ in range(50)]

# 校准
print("正在校准模型...")
activation_stats = calibrate_model(model_static, calibration_data, num_batches=30)

print(f"\n收集到 {len(activation_stats)} 层的激活值统计")
for name, stats in list(activation_stats.items())[:3]:
    print(f"  {name}: min={min(stats['min']):.4f}, max={max(stats['max']):.4f}")

In [ ]:
# 应用静态量化
static_quantizer = StaticQuantizer()
static_quantizer.calibrate(model_static, calibration_data, num_batches=30)
quantized_static = static_quantizer.quantize(model_static)

# 测试
with torch.no_grad():
    static_output = quantized_static(x_test)

# 比较
print("静态量化 vs 原始模型:")
diff = (model_static(x_test) - static_output).abs()
print(f"  平均差异: {diff.mean():.6f}")
print(f"  最大差异: {diff.max():.6f}")

## 4. 量化感知训练 (QAT)

QAT 在训练过程中模拟量化效果，使模型学习适应量化误差。

**核心技术**: 伪量化 (Fake Quantization)
- 前向传播: 量化 → 反量化
- 反向传播: 直通估计器 (STE)

In [ ]:
from quantization import FakeQuantize, FakeQuantizeModule, QATWrapper

# 演示伪量化
x = torch.randn(10, requires_grad=True)
scale = torch.tensor(0.1)
zero_point = torch.tensor(0.0)

# 伪量化
x_fake_q = FakeQuantize.apply(x, scale, zero_point, -128, 127)

print("伪量化演示:")
print(f"原始值: {x[:5].detach().numpy()}")
print(f"伪量化后: {x_fake_q[:5].detach().numpy()}")

# 验证梯度可以传播
loss = x_fake_q.sum()
loss.backward()
print(f"\n梯度: {x.grad[:5].numpy()}")
print("梯度成功传播! (STE 工作正常)")

In [ ]:
# QAT 训练示例
from quantization import quantize_model

# 创建模型并包装为 QAT
model_qat = SimpleClassifier()
qat_model = quantize_model(model_qat, quant_type="qat")

# 创建模拟训练数据
train_data = [(torch.randn(32, 784), torch.randint(0, 10, (32,))) for _ in range(100)]

# 训练
optimizer = torch.optim.Adam(qat_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("QAT 训练中...")
qat_model.train()
for epoch in range(3):
    total_loss = 0
    for x_batch, y_batch in train_data:
        optimizer.zero_grad()
        output = qat_model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"  Epoch {epoch+1}: Loss = {total_loss/len(train_data):.4f}")

print("\nQAT 训练完成!")

In [ ]:
# 转换为量化模型
qat_model.eval()
final_quantized = qat_model.convert_to_quantized()

# 测试
with torch.no_grad():
    qat_output = final_quantized(x_test)

print("QAT 模型转换完成!")
print(f"输出形状: {qat_output.shape}")

## 5. 量化方法对比

| 方法 | 校准数据 | 训练 | 精度 | 适用场景 |
|:-----|:--------:|:----:|:----:|:---------|
| 动态量化 | 不需要 | 不需要 | 中 | 快速部署 |
| 静态量化 | 需要 | 不需要 | 高 | 生产部署 |
| QAT | 需要 | 需要 | 最高 | 精度敏感 |

In [ ]:
# 总结可视化
methods = ['FP32\n(原始)', '动态量化', '静态量化', 'QAT']
model_sizes = [4, 1, 1, 1]  # 相对大小
accuracies = [100, 98, 99, 99.5]  # 假设精度

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 模型大小
colors = ['blue', 'green', 'orange', 'red']
axes[0].bar(methods, model_sizes, color=colors, alpha=0.7)
axes[0].set_ylabel('相对模型大小')
axes[0].set_title('模型大小比较')
axes[0].set_ylim(0, 5)

# 精度
axes[1].bar(methods, accuracies, color=colors, alpha=0.7)
axes[1].set_ylabel('相对精度 (%)')
axes[1].set_title('精度比较')
axes[1].set_ylim(95, 101)

plt.tight_layout()
plt.show()

print("\n量化可以将模型大小减少 4x，同时保持接近原始精度!")

## 总结

本教程介绍了模型量化的核心概念和实现：

1. **量化基础**: scale 和 zero_point 的计算
2. **动态量化**: 简单快速，无需校准
3. **静态量化**: 使用校准数据，精度更高
4. **QAT**: 训练中模拟量化，精度最高

### 选择建议

- **快速部署**: 动态量化
- **生产环境**: 静态量化
- **精度敏感**: QAT

### 下一步

- 学习模型剪枝技术
- 探索知识蒸馏
- 了解模型导出和部署